# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

Energy consumption is closely tied to weather, especially temperature. As temperatures rise or fall, heating and cooling systems drive noticeable changes in energy use. This project combines weather and energy data to analyze these patterns and better understand how temperature influences energy consumption. The research question guiding this analysis is: *How does temperature affect energy consumption patterns?

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

#### **Dataset 1**

Type: API

Method: The data was gathered programmatically using an HTTP GET request to the NOAA National Centers for Environmental Information (NCEI) Daily Summaries API. The API was queried for daily weather observations for the Cincinnati (CVG Airport) weather station over the period 2022–2023. The returned JSON response was loaded into a pandas DataFrame and stored locally as a raw CSV file before cleaning.

Dataset variables:

*   DATE: Date of the weather observation
*   TAVG: Daily average temperature (°C)

The weather dataset was selected because temperature is a primary factor influencing energy consumption, particularly for heating and cooling. This data is gathered programmatically from a public weather API and includes daily average temperature, which provides the environmental context needed to examine how changes in weather conditions relate to shifts in energy usage. These variables provide the environmental context needed to examine how changes in weather conditions relate to shifts in energy usage.

In [ ]:
# Dataset 1: Weather data gathering and loading
import pandas as pd
import requests

base_url = "https://www.ncei.noaa.gov/access/services/data/v1"
params = {
    "dataset": "daily-summaries",
    "stations": "USW00093814",
    "startDate": "2022-01-01",
    "endDate": "2023-12-31",
    "dataTypes": "TAVG",
    "units": "metric",
    "format": "json"
}

response = requests.get(base_url, params=params)
weather_df = pd.DataFrame(response.json())


#### Dataset 2

Type: CSV File

Method: The data was gathered using the manual download method from the U.S. Energy Information Administration (EIA) website.

Dataset variables:

*   city: Name of the U.S. city where energy consumption is recorded
*   state: U.S. state corresponding to each city
*   monthly energy consumption values: Numeric values representing energy usage for each month (YYYYMM format), later reshaped into a single time-based variable during cleaning

The energy consumption dataset was selected because it provides detailed, city-level records of monthly energy usage across the United States, allowing for analysis of temporal and geographic patterns in energy demand. The data was gathered via manual download from the U.S. Energy Information Administration (EIA), a reliable public data source, and then loaded programmatically into the notebook for assessment and cleaning. Key variables such as city, state, and monthly energy consumption values are essential for examining how energy usage changes over time and for aligning the data with weather observations.


In [ ]:
import pandas as pd

# Load raw energy consumption data
energy_df = pd.read_csv("data/raw/eia_energy_consumption_raw.csv")

# Preview dataset
energy_df.head()
energy_df.shape

Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
# Raw energy data is already stored locally at:
# data/raw/eia_energy_consumption_raw.csv

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1: Missing values

In [ ]:
energy_df.head()

In [ ]:
energy_df.isnull().sum().sort_values(ascending=False).head(10)

**Issue:** Several columns in the energy consumption dataset contain a substantial number of missing values, indicating a completeness issue. In particular, the `elec_renewable_source` column shows missing data for a large portion of observations.

**Justification:** Visual inspection of the dataset revealed many empty cells across certain columns, while programmatic inspection using `isnull().sum()` quantified the extent of missing values. These methods were used together to confirm both the presence and severity of the missing data and to identify columns that may need to be removed or handled during cleaning.

### Quality Issue 2: Inconsistent data types

In [ ]:
energy_df.head()

In [ ]:
energy_df.info()

**Issue:** The dataset contains inconsistent data types, with some columns storing numeric values as strings or mixed types. This is particularly evident in columns related to energy values and was confirmed by a pandas `DtypeWarning` during data loading.

**Justification:** Visual inspection helped identify values that appeared numeric but were not stored consistently, while the `info()` method provided a clear summary of column data types. These methods were used because inconsistent data types can interfere with numeric operations and must be resolved before analysis.

### Tidiness Issue 1: Wide-format monthly columns

In [ ]:
energy_df.head()

In [ ]:
energy_df.shape
energy_df.columns

**Issue:** The energy consumption dataset is not tidy because monthly energy values are stored across many separate columns, with each column representing a different month. This wide format makes time-based analysis and dataset merging difficult.

**Justification:** Visual inspection revealed dozens of month-labeled columns, while programmatic inspection of the dataset shape and column names confirmed this structure. These methods were used because tidy data principles require that each variable form a single column and each observation form a single row.

### Tidiness Issue 2: Multiple observational units stored in a single table

In [ ]:
energy_df.head()

In [ ]:
energy_df.info()

**Issue:** The energy consumption dataset contains multiple types of observational units within a single table. Static location and infrastructure attributes (such as city, state, and station metadata) are stored alongside repeated time-based energy consumption measurements, even though they represent different observational units.

**Justification:** Visual inspection of the dataset shows that identifier-level metadata and monthly measurement values coexist in the same table. Programmatic inspection of column names confirms the presence of both descriptive attributes and time-series variables. According to tidy data principles, each type of observational unit should be stored in its own table to reduce redundancy and improve clarity.

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# Make copies of the raw datasets to preserve original data
weather_clean = weather_df.copy()
energy_clean = energy_df.copy()

### Quality Issue 1: Missing values in energy consumption data

In [ ]:
# Remove column with excessive missing values
energy_clean = energy_clean.drop(columns=['elec_renewable_source'])

In [ ]:
'elec_renewable_source' in energy_clean.columns

Justification: The elec_renewable_source column was removed because it contained a substantial number of missing values and was not relevant to the research question. Removing this column improves data completeness while preserving variables necessary for analyzing energy consumption patterns. Validation confirmed the column was successfully removed.


### Quality Issue 2: Inconsistent data types in monthly energy columns

In [ ]:
# Identify monthly energy columns
monthly_cols = [col for col in energy_clean.columns if col.isdigit()]

# Convert monthly columns to numeric
energy_clean[monthly_cols] = energy_clean[monthly_cols].apply(
    pd.to_numeric, errors='coerce'
)

In [ ]:
energy_clean[monthly_cols].dtypes.value_counts()

Justification: Unnecessary variables unrelated to the research question were removed to simplify the dataset structure. Retaining only location identifiers and energy values improves clarity and usability. Validation confirmed the reduced dataset structure.

### Tidiness Issue 1: Monthly values stored in wide format

In [ ]:
energy_long = energy_clean.melt(
    id_vars=['city', 'state'],
    value_vars=monthly_cols,
    var_name='year_month',
    value_name='energy_consumption'
)

In [ ]:
energy_long.head()
energy_long.dtypes
energy_long.shape

Justification: The dataset was reshaped from wide to long format to comply with tidy data principles, ensuring that each variable forms a column and each observation forms a row. Time information was standardized into a single datetime column to support temporal analysis and dataset merging.

### Tidiness Issue 2: Multiple observational units stored in a single table

In [ ]:
# Clean Tidiness Issue 2: Remove facility metadata columns
# The melt operation above (Tidiness Issue 1) with id_vars=['city', 'state']
# already filtered out facility-related columns (name, address, facility_type, etc.)
# This focused the dataset on a single observational unit: energy consumption by location

print("Columns in energy_long after removing facility metadata:")
print(energy_long.columns.tolist())
print(f"\nDataset now contains only: {len(energy_long.columns)} columns")
energy_long.head()

Justification: The original energy dataset contained two observational units: (1) facility characteristics (name, street address, facility type, network key) and (2) energy consumption measurements. By specifying only `city` and `state` as id_vars in the melt operation, all facility metadata columns were automatically excluded. The resulting dataset now represents a single observational unit—energy consumption by location over time—which aligns with tidy data principles.

In [ ]:
# Validate: Confirm facility columns are gone
print("Sample of data - only location and consumption remain:")
print(energy_long.info())

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
# Convert DATE column to datetime
weather_clean['DATE'] = pd.to_datetime(weather_clean['DATE'])

# Convert temperature to numeric
weather_clean['TAVG'] = pd.to_numeric(weather_clean['TAVG'], errors='coerce')

# Create year_month column for aggregation
weather_clean['year_month'] = weather_clean['DATE'].dt.to_period('M').dt.to_timestamp()

# Aggregate daily temperatures to monthly averages
weather_monthly = (
    weather_clean
    .groupby('year_month', as_index=False)
    .agg(avg_temperature=('TAVG', 'mean'))
)

# Preview monthly weather data
weather_monthly.head()
weather_monthly.shape

In [ ]:
# Ensure BOTH datasets use the same datetime type for year_month
energy_long['year_month'] = pd.to_datetime(
    energy_long['year_month'],
    format='%Y%m'
)

weather_monthly['year_month'] = pd.to_datetime(
    weather_monthly['year_month']
)

# Keep only relevant weather variables (preprocessing)
weather_tidy = weather_monthly[['year_month', 'avg_temperature']]

# Combine energy and weather datasets on year_month
final_df = energy_long.merge(
    weather_tidy,
    on='year_month',
    how='inner'
)

# Preview combined dataset
final_df.head()
final_df.shape

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
import os

# Ensure cleaned data directory exists
os.makedirs("data/cleaned", exist_ok=True)

# Save cleaned datasets
weather_monthly.to_csv(
    "data/cleaned/weather_cincinnati_monthly_clean.csv",
    index=False
)

energy_long.to_csv(
    "data/cleaned/energy_consumption_cleaned.csv",
    index=False
)

final_df.to_csv(
    "data/cleaned/energy_weather_monthly_combined.csv",
    index=False
)

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

Research question: How does temperature affect energy consumption patterns?

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(
    final_df['avg_temperature'],
    final_df['energy_consumption']
)
plt.xlabel("Average Monthly Temperature (°C)")
plt.ylabel("Monthly Energy Consumption")
plt.title("Energy Consumption vs Temperature")
plt.show()

Answer to research question: The scatter plot shows that energy consumption varies more widely at lower and higher average temperatures. This suggests that energy usage increases during colder and warmer months due to heating and cooling demands, while moderate temperatures are associated with lower and more stable energy consumption.


In [ ]:
monthly_summary = (
    final_df
    .groupby('year_month', as_index=False)
    .agg({
        'energy_consumption': 'mean',
        'avg_temperature': 'mean'
    })
)

monthly_summary.head()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(
    monthly_summary['year_month'],
    monthly_summary['energy_consumption'],
    label='Energy Consumption'
)
ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Energy Consumption")

ax2 = ax1.twinx()
ax2.plot(
    monthly_summary['year_month'],
    monthly_summary['avg_temperature'],
    label='Average Temperature'
)
ax2.set_ylabel("Average Temperature (°C)")

plt.title("Monthly Energy Consumption and Temperature Over Time")
plt.show()

Answer to research question: The time-series visualization shows clear seasonal patterns in both temperature and energy consumption. Energy usage remains elevated during periods of extreme temperatures, particularly in colder and warmer months, indicating increased demand for heating and cooling. This temporal alignment supports the conclusion that temperature plays a significant role in shaping energy consumption patterns.

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

Answer: With more time, this project could be expanded by incorporating weather data from multiple regions instead of a single metropolitan area to better capture geographic variation in energy consumption. Additional energy variables, such as sector-specific or residential versus commercial usage, could also improve the analysis. Further data quality checks on regional alignment and temporal granularity would strengthen future research.
